# ERA5 day-of-year climatology v2 — detrended σ + incremental cities

This v2 notebook does two things differently from v1:

**1. Detrended σ (anomaly method).** For each city, we fit a smooth seasonal cycle (low-order Fourier harmonics) to the 30 years of daily-mean temperatures, subtract it to obtain anomalies, and compute σ from those anomalies within each 15-day window. This removes the bias from rapid spring/fall warming inflating σ artificially. The mean stays the same (it's just the harmonic fit value).

**2. Incremental city append.** If you already have CSVs in your Drive from the v1 notebook (the original 105 cities), this notebook only exports new cities and merges with the existing data. Saves ~80% of GEE compute time on a re-run.

**Output.** `era5_doy_clim.parquet` with columns `(geonames_id, doy, mean_c, std_c, p95_c, p99_c, n)` — 366 DOYs × ~312 cities ≈ 114 k rows.

**Run when.** Re-run when you change the city list, or when WMO updates the standard normal period (next due 2031).

In [ ]:
# Cell 1 — install + auth
!pip install -q earthengine-api pandas pyarrow

import ee
ee.Authenticate()
ee.Initialize(project='YOUR-EE-PROJECT-ID')   # ← your EE Cloud project ID

In [ ]:
# Cell 2 — load city list, identify which cities still need GEE export
import pandas as pd
from google.colab import files, drive
import glob, os

# Upload pipeline/cities.csv
uploaded = files.upload()
cities = pd.read_csv(next(iter(uploaded.keys())))
print(f'Total cities in list: {len(cities)}')

# Mount Drive to inspect what was already exported in v1
drive.mount('/content/drive')
existing_paths = sorted(glob.glob('/content/drive/MyDrive/heatmap_climatology/era5_samples_*.csv'))
print(f'Found {len(existing_paths)} existing year files from v1 (if any)')

if existing_paths:
    existing_df = pd.concat([pd.read_csv(p) for p in existing_paths], ignore_index=True)
    existing_ids = set(existing_df['geonames_id'].unique())
    new_ids = set(cities['geonames_id']) - existing_ids
    new_cities = cities[cities['geonames_id'].isin(new_ids)]
    print(f'Already have data for {len(existing_ids)} cities')
    print(f'Need to export {len(new_cities)} new cities to GEE')
else:
    new_cities = cities
    existing_df = None
    print(f'No existing data — exporting all {len(new_cities)} cities')

In [ ]:
# Cell 3 — config
START_YEAR = 1991
END_YEAR = 2020
WINDOW_DAYS = 15
HALF_WINDOW = WINDOW_DAYS // 2
N_HARMONICS = 4   # # of Fourier harmonics for the seasonal-cycle fit

ERA5 = ee.ImageCollection('ECMWF/ERA5/DAILY').select('mean_2m_air_temperature')

if len(new_cities) == 0:
    print('Nothing to export — proceed directly to Cell 6')
else:
    feats = [
        ee.Feature(
            ee.Geometry.Point([row['lon'], row['lat']]),
            {'geonames_id': int(row['geonames_id']), 'name': row['name']}
        )
        for _, row in new_cities.iterrows()
    ]
    city_fc = ee.FeatureCollection(feats)
    print(f'Built FC with {len(feats)} new city points')

In [ ]:
# Cell 4 — kick off per-year exports for the NEW cities only
# Files land at: MyDrive/heatmap_climatology/era5_samples_NEW_<year>.csv
# (NEW prefix avoids overwriting v1 files)

if len(new_cities) == 0:
    print('No new cities to export — skip this cell.')
else:
    def sample_year(year):
        ic = ERA5.filterDate(f'{year}-01-01', f'{year + 1}-01-01')
        def per_image(img):
            date = img.date().format('YYYY-MM-dd')
            sampled = img.reduceRegions(
                collection=city_fc,
                reducer=ee.Reducer.first(),
                scale=27830
            )
            return sampled.map(lambda f: f.set('date', date))
        return ic.map(per_image).flatten()

    tasks = []
    for year in range(START_YEAR, END_YEAR + 1):
        fc = sample_year(year)
        task = ee.batch.Export.table.toDrive(
            collection=fc,
            description=f'era5_samples_NEW_{year}',
            folder='heatmap_climatology',
            fileNamePrefix=f'era5_samples_NEW_{year}',
            fileFormat='CSV',
            selectors=['geonames_id', 'date', 'first']
        )
        task.start()
        tasks.append((year, task))
        print(f'Started export for {year}: {task.id}')
    print(f'\nKicked off {len(tasks)} exports. Monitor at https://code.earthengine.google.com/tasks')

**Wait for the 30 NEW exports to finish in the EE Tasks panel.** ~1-3 hours wall-clock. You can close this notebook; results persist in Drive.

In [ ]:
# Cell 5 — load + concatenate v1 + new exports
import pandas as pd, glob

all_paths = sorted(glob.glob('/content/drive/MyDrive/heatmap_climatology/era5_samples_*.csv'))
print(f'Found {len(all_paths)} CSV files (v1 + new merged)')

df = pd.concat([pd.read_csv(p) for p in all_paths], ignore_index=True)
df = df.rename(columns={'first': 'temp_k'})
df['temp_c'] = df['temp_k'] - 273.15
df['date'] = pd.to_datetime(df['date'])
df['doy'] = df['date'].dt.dayofyear
df.loc[(df['date'].dt.month == 2) & (df['date'].dt.day == 29), 'doy'] = 59

# Subset to cities still in the current list (drops anyone removed)
df = df[df['geonames_id'].isin(cities['geonames_id'])]
print(f'Total samples: {len(df):,} across {df.geonames_id.nunique()} cities')

In [ ]:
# Cell 6 — DETRENDED-SIGMA climatology
# For each city: fit a smooth seasonal cycle of daily-mean temp (low-order
# Fourier in DOY); compute residuals; for each DOY take the mean of the
# fitted seasonal cycle within the 15-day window (= mean_c) and the σ of
# residuals within the window (= std_c). This σ measures *interannual*
# variability without inflation from rapid seasonal change.
import numpy as np

def fit_seasonal(doys, temps, n_harm=N_HARMONICS):
    """Least-squares fit of mean + sin/cos harmonics on DOY."""
    d = 2 * np.pi * doys / 365.25
    cols = [np.ones_like(d)]
    for k in range(1, n_harm + 1):
        cols.append(np.cos(k * d))
        cols.append(np.sin(k * d))
    X = np.column_stack(cols)
    coef, *_ = np.linalg.lstsq(X, temps, rcond=None)
    return coef

def evaluate_seasonal(coef, doys, n_harm=N_HARMONICS):
    d = 2 * np.pi * np.asarray(doys) / 365.25
    out = np.full_like(d, coef[0], dtype=float)
    for k in range(1, n_harm + 1):
        out += coef[2 * k - 1] * np.cos(k * d)
        out += coef[2 * k]     * np.sin(k * d)
    return out

def doy_window(doy, half=HALF_WINDOW):
    return [((doy - 1 + d) % 365) + 1 for d in range(-half, half + 1)]

rows = []
for gid, sub in df.groupby('geonames_id', sort=False):
    doys = sub['doy'].values
    temps = sub['temp_c'].values
    coef = fit_seasonal(doys, temps)
    seasonal = evaluate_seasonal(coef, doys)
    anomalies = temps - seasonal
    # Bucket anomalies by DOY for quick window lookups
    anom_by_doy = pd.Series(anomalies).groupby(sub['doy'].values).apply(list).to_dict()
    for doy in range(1, 366):
        in_window = []
        for d in doy_window(doy):
            in_window.extend(anom_by_doy.get(d, []))
        if len(in_window) < 100:
            print(f'WARN: gid={gid} doy={doy} only {len(in_window)} samples')
        arr = np.array(in_window)
        mean_c = float(evaluate_seasonal(coef, [doy])[0])
        std_c  = float(arr.std(ddof=1))
        p95_c  = mean_c + float(np.percentile(arr, 95))
        p99_c  = mean_c + float(np.percentile(arr, 99))
        rows.append({
            'geonames_id': int(gid), 'doy': doy,
            'mean_c': mean_c, 'std_c': std_c,
            'p95_c': p95_c, 'p99_c': p99_c,
            'n': int(len(arr))
        })

clim = pd.DataFrame(rows)
print(clim.describe())
print(f'\nClimatology rows: {len(clim):,}')

In [ ]:
# Cell 7 — sanity-check the detrended σ
# For a midlatitude city, σ in v1 (raw) was inflated near equinoxes;
# in v2 (detrended) σ should be much flatter through the year.
import matplotlib.pyplot as plt

test_gids = [3530597, 1275339, 2147714, 524901, 2660646]   # Mexico City, Mumbai, Sydney, Moscow, Geneva
fig, axes = plt.subplots(len(test_gids), 1, figsize=(10, 2 * len(test_gids)), sharex=True)
for ax, gid in zip(axes, test_gids):
    sub = clim[clim['geonames_id'] == gid].sort_values('doy')
    name = cities.loc[cities['geonames_id'] == gid, 'name'].iloc[0] if gid in cities['geonames_id'].values else str(gid)
    ax.plot(sub['doy'], sub['std_c'], lw=1.5, color='C3')
    ax.set_ylabel('σ (°C)')
    ax.set_title(f'{name}: detrended σ by DOY')
    ax.grid(alpha=0.3)
axes[-1].set_xlabel('Day of year')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — write parquet and download
out_path = '/content/era5_doy_clim.parquet'
clim.to_parquet(out_path, index=False)
print(f'Wrote {out_path} — {clim.shape[0]:,} rows')

from google.colab import files
files.download(out_path)
# Then in Terminal:
#   mv ~/Downloads/era5_doy_clim.parquet ~/GitHub/whereishot/climatology/
#   git add climatology/era5_doy_clim.parquet
#   git commit -m 'Climatology v2: detrended σ + 312 cities'
#   git push